In [25]:
import os
import cv2
import numpy as np

# ---------------------
# PATHS (edit if needed)
# ---------------------
IMG_DIR = "C:/Users/sanja/Desktop/MRI_Project/Pleural Effusion/Dataset"          # original images
OUT_DIR = "C:/Users/sanja/Desktop/MRI_Project/Pleural Effusion/AUG_Dataset"       # new augmented dataset folder

os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# FIXED, SAFE AUGMENTATION
# -------------------------
class CTAugment:
    def __init__(self):
        pass

    def __call__(self, image):
        # convert to float32 to avoid overflow
        image = image.astype("float32")
        h, w = image.shape

        # rotation
        angle = np.random.uniform(-12, 12)
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        image = cv2.warpAffine(image, M, (w, h))

        # horizontal flip
        if np.random.rand() < 0.5:
            image = cv2.flip(image, 1)

        # brightness shift
        shift = np.random.uniform(-15, 15)
        image = image + shift

        # blur
        if np.random.rand() < 0.5:
            image = cv2.GaussianBlur(image, (3,3), 0)

        # zoom via crop
        pct = np.random.uniform(0.02, 0.08)
        ch, cw = int(h*pct), int(w*pct)
        cropped = image[ch:h-ch, cw:w-cw]
        image = cv2.resize(cropped, (w, h))

        # final clamp
        image = np.clip(image, 0, 255).astype("uint8")

        return image


augment = CTAugment()

# -------------------------
# GENERATE 400 IMAGES
# -------------------------
paths = [f for f in os.listdir(IMG_DIR) if f.endswith(".PNG")]
repeats = 70   # 6 × 70 = 420 images
save_counter = 1

for img_name in paths:
    img_path = os.path.join(IMG_DIR, img_name)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # generate 70 augmented versions
    for i in range(repeats):
        aug_img = augment(img.copy())

        out_name = f"aug_{save_counter:04d}.png"
        out_path = os.path.join(OUT_DIR, out_name)

        cv2.imwrite(out_path, aug_img)
        save_counter += 1

print("DONE! Augmented images saved in:", OUT_DIR)
print("Total images generated:", save_counter - 1)


DONE! Augmented images saved in: C:/Users/sanja/Desktop/MRI_Project/Pleural Effusion/AUG_Dataset
Total images generated: 420
